In [ ]:
!pip install rdflib

In [ ]:
import json
from rdflib import Graph
from rdflib.plugins.sparql import prepareQuery

# Configuration - edit these paths directly
INPUT_JSON_PATH = "../evaluation/llm_as_a_judge/input/json_files/rdf_extractions_oneshot.json"
OUTPUT_JSON_PATH = "rdf_validation_oneshot.json"


def check_rdf_executability(turtle_text: str) -> dict:
    """
    Parse Turtle RDF content and verify basic executability.
    
    Args:
        turtle_text: String containing Turtle RDF content
        
    Returns:
        dict: Result dictionary with parsing status and counts
    """
    result = {
        "RDF syntax": "is invalid",
        "parsed": "failed", 
        "Number of triples in RDF": 0,
        "skos:Concepts Found": 0
    }
    
    # Create a new RDF graph
    g = Graph()
    
    try:
        # Parse the RDF data
        g.parse(data=turtle_text, format="turtle")
        
        # If parsing succeeds, update result
        result["RDF syntax"] = "is valid"
        result["parsed"] = "successfully"
        result["Number of triples in RDF"] = len(g)
        
    except Exception as e:
        # Parsing failed - return result with default values
        return result
    
    try:
        # Run SPARQL query to count skos:Concept instances
        # Use full URI to avoid namespace prefix dependencies
        query = prepareQuery("""
            SELECT (COUNT(?c) AS ?count) WHERE {
                ?c a <http://www.w3.org/2004/02/skos/core#Concept> .
            }
        """)
        
        query_results = g.query(query)
        
        # Extract count from query results
        for row in query_results:
            result["skos:Concepts Found"] = int(row[0])
            break
            
    except Exception as e:
        # SPARQL query failed - keep concept count at 0
        pass
    
    return result


def load_input_json(input_path: str) -> dict:
    """
    Load JSON input file.
    
    Args:
        input_path: Path to input JSON file
        
    Returns:
        dict: Loaded JSON data
    """
    with open(input_path, 'r', encoding='utf-8') as f:
        return json.load(f)


def save_output_json(output_path: str, data: dict) -> None:
    """
    Save data to JSON output file.
    
    Args:
        output_path: Path to output JSON file
        data: Dictionary to save as JSON
    """
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def main():
    """
    Main processing function that loads input, processes RDF, and saves output.
    """
    try:
        # Load input JSON
        input_data = load_input_json(INPUT_JSON_PATH)
        
        # Process each dataset entry
        all_results = []
        
        for entry in input_data.get("dataset", []):
            # Extract turtle content and source image
            turtle_content = entry.get("rdf_graph_turtle", "")
            source_image = entry.get("source_image", "")
            
            # Check RDF executability
            result = check_rdf_executability(turtle_content)
            
            # Add source image to result
            result["source_image"] = source_image
            
            all_results.append(result)
        
        # Save all results to output JSON
        output_data = {"results": all_results}
        save_output_json(OUTPUT_JSON_PATH, output_data)
        
        # Optional: print results summary
        print(f"Processing complete. Processed {len(all_results)} RDF entries.")
        print(f"Results saved to: {OUTPUT_JSON_PATH}")
        
        # Print individual results
        for i, result in enumerate(all_results, 1):
            print(f"\nEntry {i}: {result['source_image']}")
            print(f"  RDF syntax: {result['RDF syntax']}")
            print(f"  Parsed: {result['parsed']}")
            print(f"  Triples: {result['Number of triples in RDF']}")
            print(f"  SKOS Concepts: {result['skos:Concepts Found']}")
        
    except Exception as e:
        # Handle any unexpected errors
        error_result = {
            "error": str(e),
            "results": []
        }
        save_output_json(OUTPUT_JSON_PATH, error_result)
        print(f"Error during processing: {e}")


if __name__ == "__main__":
    main()